# TrainLM Colab v5e-1 Full Validation Suite

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_Colab_v5e1_Full_Validation.ipynb)

Run this notebook on a **Colab runtime connected to a single-chip Cloud TPU v5e (`ct5lp-hightpu-1t`)** to exercise every currently implemented TrainLM feature and hardware gate in one pass, then download a single sanitized JSON evidence report.

**Why subprocess cells run before in-process cells:** on single-chip Cloud TPU v5e-1, whichever Python process first initializes `torch_tpu` owns the TPU device for its lifetime. Cells 3–5 run isolated child-process checks (`T1` probes, `validate_hf_pretraining.py`, and `trainlm` CLI) so each child releases the device cleanly before Cells 6–9 run in-process API checks.

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, distribution, version
from pathlib import Path

BRANCH = "main"  # Set to a PR branch name when testing pre-merge changes.
REPO_URL = "https://github.com/Dhiraj309/TrainLM.git"

# Install TrainLM without touching preinstalled PyTorch, TorchTPU, or libtpu.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", f"trainlm @ git+{REPO_URL}@{BRANCH}"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "huggingface-hub>=1.5,<2",
        "PyYAML>=6,<7",
        "transformers>=5.15,<6",
        "datasets>=3,<5",
        "tokenizers>=0.21,<1",
    ],
    check=True,
)

WORK_ROOT = Path(tempfile.mkdtemp(prefix="trainlm-v5e1-full-"))
REPO_ROOT = WORK_ROOT / "TrainLM"
subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_ROOT)],
    check=True,
    timeout=120,
)
SOURCE_REVISION = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", SOURCE_REVISION], check=True, timeout=30)

direct_url = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
INSTALLED_REVISION = direct_url.get("vcs_info", {}).get("commit_id")
NOTEBOOK_HOST = (
    "Colab" if ("COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules)
    else "Kaggle" if ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ)
    else "other"
)
PACKAGES = {}
for pkg in ("trainlm", "torch", "torch-tpu", "libtpu", "transformers", "datasets", "tokenizers", "huggingface-hub"):
    try:
        PACKAGES[pkg] = version(pkg)
    except PackageNotFoundError:
        PACKAGES[pkg] = None

print(f"notebook_host={NOTEBOOK_HOST} branch={BRANCH}")
print(f"source_revision={SOURCE_REVISION} installed_revision={INSTALLED_REVISION}")
print("packages=", json.dumps(PACKAGES, indent=2))

## Validation Control Panel

Edit the toggles below (or leave defaults) and choose **Runtime → Run all**. All sections catch failures independently so the final evidence JSON is always produced and downloadable.

In [ ]:
import runpy
import time
import traceback

# Hardware metadata (enter 'ct5lp-hightpu-1t' if confirmed in Cloud TPU settings).
TPU_VM_MACHINE_TYPE = None

# Subprocess checks (Cells 3-5): release TPU on exit.
RUN_T1_EAGER = True
RUN_T1_COMPILE = True          # Tests synthetic eager-vs-compiled parity on v5e-1.
RUN_T1_LIFECYCLE = True        # Tests 2 sequential fresh TPU child processes + teardown.
T1_LIFECYCLE_COMPILED = False  # Set True after RUN_T1_COMPILE passes.
RUN_GPT2_HF_MATRIX = True      # 4-case matrix (random/pretrained x raw/tokens) for GPT-2.
RUN_LLAMA_HF_MATRIX = True     # 4-case matrix for Llama architecture cross-check.
RUN_CLI_WORKFLOWS = True       # trainlm pretrain/finetune (--dry-run and real native_tpu run).

# In-process checks (Cells 6-9): run inside the notebook kernel on native_tpu.
PRECISION_MODE = "bf16"        # "bf16", "fp32", or "both"
TEST_PRETRAIN_OPTIONS = True   # cosine, warmup_ratio, grad checkpointing, best model, early stop, formatter, MFU.
TEST_PRETRAIN_COMPILE = False  # Set True to test torch_compile=True inside run_pretraining.
TEST_VM_RECOVERY = True        # bundle(include_resume_state=True) -> new dir -> restore -> resume + extend.
TEST_FINETUNING_OPTIONS = True # Finite Hub, local JSONL epoch+best+early-stop, pretokenized + vocab SHA.
TEST_FINETUNING_STREAMING = True  # Hub streaming=True + exact cursor restore/resume.
TEST_EPOCH_AND_CHUNKED_LOSS = True  # PackedBinDataset epoch schedules + chunked_linear training_view.
RUN_MFU_BASELINE = True        # Benchmark SmolLM2-135M (eager+compile), SmolLM2-360M, and Qwen2.5-0.5B.
MFU_RUN_COMPILED = True        # Compare eager vs torch_compile=True on SmolLM2-135M.

# Optional live Hub upload check (Cell 10). Leave None to skip.
HUB_REPO_ID = None             # e.g. "your-username/trainlm-v5e1-smoke"
HUB_PRIVATE = True

RESULTS = {}
LAST_EXPORT_RESULT = None
RUN_PROCESS = runpy.run_path(str(REPO_ROOT / "src/trainlm/runtime/torchtpu_probe_process.py"))["run_probe_process"]
EVIDENCE = runpy.run_path(str(REPO_ROOT / "src/trainlm/runtime/torchtpu_evidence.py"))


def record_check(name, enabled, fn):
    if not enabled:
        RESULTS[name] = {"status": "not_run", "reason": "disabled_by_toggle"}
        print(f"[SKIP] {name}")
        return None
    started = time.perf_counter()
    print(f"[RUN ] {name} ...", flush=True)
    try:
        details = fn()
        elapsed = round(time.perf_counter() - started, 3)
        RESULTS[name] = {"status": "passed", "elapsed_seconds": elapsed, "details": details or {}}
        print(f"[PASS] {name} ({elapsed}s)", flush=True)
        return details
    except Exception as exc:
        elapsed = round(time.perf_counter() - started, 3)
        RESULTS[name] = {
            "status": "failed",
            "elapsed_seconds": elapsed,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        }
        print(f"[FAIL] {name} ({elapsed}s): {type(exc).__name__}: {exc}", flush=True)
        traceback.print_exc()
        return None


def verify_cpu_export_generation(export_path):
    proc = subprocess.run(
        [sys.executable, str(REPO_ROOT / "examples/infer_local_export.py"), str(export_path), "--max-new-tokens", "2"],
        cwd=REPO_ROOT, text=True, capture_output=True, check=True, timeout=180,
    )
    return proc.stdout.strip()

## Section 1 (Subprocesses): T1 Package Preflight, `trainlm doctor`, Eager/Compile & Lifecycle Probes

In [ ]:
def _run_preflight():
    res = RUN_PROCESS([sys.executable, str(REPO_ROOT / "examples/torchtpu_t1_install_preflight.py")], cwd=REPO_ROOT, timeout=60)
    ev = EVIDENCE["profile_evidence"](res)
    if res["exit_code"] not in (0, 1):
        raise RuntimeError(f"preflight exited {res['exit_code']}")
    return ev


def _run_doctor():
    res = RUN_PROCESS([sys.executable, "-m", "trainlm", "doctor"], cwd=REPO_ROOT, timeout=300)
    payload = json.loads(res["stdout"].splitlines()[-1])
    return {"exit_code": res["exit_code"], "report_status": payload.get("status"), "accelerator": payload.get("accelerator")}


def _run_t1_probe(compiled):
    args = ["--device", "auto"] + (["--compile"] if compiled else [])
    res = RUN_PROCESS([sys.executable, str(REPO_ROOT / "examples/torchtpu_t1_v5e1_probe.py"), *args], cwd=REPO_ROOT, timeout=900 if compiled else 300)
    ev = EVIDENCE["smoke_evidence"](res, compiled=compiled)
    if ev["status"] != "passed":
        raise RuntimeError(f"T1 probe status={ev['status']} stderr={res['stderr'][-400:]}")
    return ev


def _run_t1_lifecycle():
    run_lc = runpy.run_path(str(REPO_ROOT / "examples/torchtpu_t1_lifecycle_probe.py"))["run_lifecycle"]
    rep = run_lc(WORK_ROOT / "t1-lifecycle", repeats=2, timeout=300, compiled=T1_LIFECYCLE_COMPILED)
    ev = EVIDENCE["lifecycle_evidence"](rep)
    if ev["status"] != "passed":
        raise RuntimeError(f"T1 lifecycle status={ev['status']}")
    return ev


record_check("t1_install_preflight", True, _run_preflight)
record_check("cli_doctor", True, _run_doctor)
record_check("t1_eager_probe", RUN_T1_EAGER, lambda: _run_t1_probe(False))
record_check("t1_compile_probe", RUN_T1_COMPILE, lambda: _run_t1_probe(True))
record_check("t1_lifecycle_probe", RUN_T1_LIFECYCLE, _run_t1_lifecycle)

## Section 2 (Subprocesses): 4-Case HF Pretraining/CPT Matrix (GPT-2 & Llama)

Runs `examples/validate_hf_pretraining.py --accelerator native_tpu` covering `random/pretrained` × `text/tokens` (including `tokenizer_vocab_sha256` verification and fresh-process CPU generation).

In [ ]:
def _run_hf_matrix(name, model_id):
    out_dir = WORK_ROOT / f"hf-matrix-{name}"
    subprocess.run(
        [
            sys.executable, str(REPO_ROOT / "examples/validate_hf_pretraining.py"),
            "--accelerator", "native_tpu",
            "--model", model_id,
            "--output-dir", str(out_dir),
        ],
        cwd=REPO_ROOT, check=True, timeout=900,
    )
    matrix = json.loads((out_dir / "matrix.json").read_text(encoding="utf-8"))
    if len(matrix.get("cases", {})) != 4:
        raise RuntimeError(f"Expected 4 matrix cases, got {list(matrix.get('cases', {}))}")
    return {
        "model": matrix.get("model"),
        "model_revision": matrix.get("model_revision"),
        "dataset_revision": matrix.get("dataset_revision"),
        "cases": {k: {"eval_loss": v["eval_loss"], "tokens": v["supervised_tokens"]} for k, v in matrix["cases"].items()},
    }


record_check("hf_matrix_gpt2", RUN_GPT2_HF_MATRIX, lambda: _run_hf_matrix("gpt2", "hf-internal-testing/tiny-random-gpt2"))
record_check("hf_matrix_llama", RUN_LLAMA_HF_MATRIX, lambda: _run_hf_matrix("llama", "hf-internal-testing/tiny-random-LlamaForCausalLM"))

## Section 3 (Subprocesses): `trainlm pretrain` and `trainlm finetune` CLI Workflows

In [ ]:
import yaml


def _run_cli_workflows():
    cli_dir = WORK_ROOT / "cli-checks"
    cli_dir.mkdir(parents=True, exist_ok=True)
    train_jsonl = cli_dir / "train.jsonl"
    val_jsonl = cli_dir / "val.jsonl"
    train_jsonl.write_text("\n".join(json.dumps({"text": f"Once upon a time story {i} ended happily."}) for i in range(16)) + "\n", encoding="utf-8")
    val_jsonl.write_text("\n".join(json.dumps({"text": f"Validation story {i} in the woods."}) for i in range(4)) + "\n", encoding="utf-8")

    pretrain_yaml = cli_dir / "pretrain.yaml"
    pretrain_yaml.write_text(yaml.safe_dump({
        "output_dir": str(cli_dir / "pretrain-run"),
        "model_name_or_path": "hf-internal-testing/tiny-random-gpt2",
        "model_initialization": "random",
        "train_file": str(train_jsonl),
        "validation_file": str(val_jsonl),
        "max_tokens": 90,
        "sequence_length": 16,
        "per_device_train_batch_size": 2,
        "repeat_train_data": True,
        "accelerator": "native_tpu",
        "bf16": PRECISION_MODE != "fp32",
    }), encoding="utf-8")

    finetune_yaml = cli_dir / "finetune.yaml"
    finetune_yaml.write_text(yaml.safe_dump({
        "model_name_or_path": "sshleifer/tiny-gpt2",
        "train_file": str(train_jsonl),
        "validation_file": str(val_jsonl),
        "args": {
            "output_dir": str(cli_dir / "finetune-run"),
            "max_steps": 2,
            "sequence_length": 32,
            "eval_strategy": "no",
            "save_strategy": "no",
            "report_to": "jsonl",
            "accelerator": "native_tpu",
            "bf16": PRECISION_MODE != "fp32",
        },
    }), encoding="utf-8")

    dry_p = subprocess.run([sys.executable, "-m", "trainlm", "pretrain", "--config", str(pretrain_yaml), "--dry-run"], check=True, capture_output=True, text=True, timeout=120)
    run_p = subprocess.run([sys.executable, "-m", "trainlm", "pretrain", "--config", str(pretrain_yaml)], check=True, capture_output=True, text=True, timeout=600)
    dry_f = subprocess.run([sys.executable, "-m", "trainlm", "finetune", "--config", str(finetune_yaml), "--dry-run"], check=True, capture_output=True, text=True, timeout=120)
    run_f = subprocess.run([sys.executable, "-m", "trainlm", "finetune", "--config", str(finetune_yaml)], check=True, capture_output=True, text=True, timeout=600)
    return {
        "pretrain_dry_run_workflow": json.loads(dry_p.stdout)["workflow"],
        "pretrain_summary": run_p.stdout.strip().splitlines()[-1],
        "finetune_dry_run_workflow": json.loads(dry_f.stdout)["workflow"],
        "finetune_summary": run_f.stdout.strip().splitlines()[-1],
    }


record_check("cli_pretrain_and_finetune", RUN_CLI_WORKFLOWS, _run_cli_workflows)

## Section 4 (In-Process `native_tpu`): Pretraining Options Matrix

Tests `training_plan(inspect_model=True)`, `bf16` / `fp32`, optional `torch_compile=True`, `gradient_checkpointing=True`, `lr_scheduler_type="wsd"`, `warmup_ratio`, `betas`/`max_grad_norm`, `save_total_limit`, `load_best_model_at_end`, `early_stopping_patience`, local multi-column JSONL + `formatting_func`, `repeat_train_data=True`, and measured `performance` / MFU.

In [ ]:
import torch
from trainlm import TrainLMPretrainingConfig, run_pretraining
from trainlm.runtime.torchtpu_discovery import discover_tpu, normalize_worker_report


def _run_pretrain_options(*, use_bf16, compile_enabled, tag):
    global LAST_EXPORT_RESULT
    data_dir = WORK_ROOT / f"pretrain-{tag}-data"
    data_dir.mkdir(parents=True, exist_ok=True)
    train_file = data_dir / "train.jsonl"
    val_file = data_dir / "val.jsonl"
    train_file.write_text("\n".join(json.dumps({"prompt": f"Q{i}", "answer": f"Story number {i} has a sunny ending."}) for i in range(24)) + "\n", encoding="utf-8")
    val_file.write_text("\n".join(json.dumps({"prompt": f"VQ{i}", "answer": f"Validation text {i} stays held out."}) for i in range(6)) + "\n", encoding="utf-8")

    def fmt(row):
        return f"Prompt: {row['prompt']}\nAnswer: {row['answer']}"

    cfg = TrainLMPretrainingConfig(
        output_dir=WORK_ROOT / f"pretrain-{tag}-out",
        model_name_or_path="hf-internal-testing/tiny-random-gpt2",
        model_initialization="random",
        train_file=train_file,
        validation_file=val_file,
        text_formatter_name="qa-v1",
        repeat_train_data=True,
        max_tokens=310,
        sequence_length=32,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=1,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        betas=(0.9, 0.95),
        eps=1e-8,
        max_grad_norm=1.0,
        save_steps=2,
        eval_steps=2,
        save_total_limit=2,
        load_best_model_at_end=True,
        early_stopping_patience=1,
        early_stopping_threshold=0.0,
        gradient_checkpointing=True,
        logging_steps=1,
        accelerator="native_tpu",
        bf16=use_bf16,
        torch_compile=compile_enabled,
    )
    plan = cfg.training_plan(inspect_model=True)
    res = run_pretraining(cfg, formatting_func=fmt, show_progress=True, source_revision=SOURCE_REVISION)
    gen_text = verify_cpu_export_generation(res.export_path)
    LAST_EXPORT_RESULT = res
    perf = res.report.get("training", {}).get("performance", {})
    return {
        "parameter_count_preview": plan["model"].get("parameter_count"),
        "parameter_count_actual": res.parameter_count,
        "summary": res.summary(),
        "best_model_selection": res.best_model_selection,
        "early_stopping_triggered": res.training.get("early_stopping_triggered"),
        "performance": {k: perf.get(k) for k in ("status", "tokens_per_second", "dataloader_wait_fraction", "bottleneck_hint", "mfu_status", "non_embedding_mfu", "mfu_unavailable_reason")},
        "generation_preview": gen_text[:120],
    }


if PRECISION_MODE in {"bf16", "both"}:
    record_check("pretrain_options_bf16", TEST_PRETRAIN_OPTIONS, lambda: _run_pretrain_options(use_bf16=True, compile_enabled=False, tag="bf16"))
if PRECISION_MODE in {"fp32", "both"}:
    record_check("pretrain_options_fp32", TEST_PRETRAIN_OPTIONS, lambda: _run_pretrain_options(use_bf16=False, compile_enabled=False, tag="fp32"))
record_check("pretrain_compile_bf16", TEST_PRETRAIN_COMPILE, lambda: _run_pretrain_options(use_bf16=True, compile_enabled=True, tag="compile"))

try:
    RESULTS["in_process_tpu_discovery"] = normalize_worker_report(discover_tpu(torch, verify_allocation=False, include_properties=True))
except Exception as exc:
    RESULTS["in_process_tpu_discovery"] = {"status": "query_failed", "error_type": type(exc).__name__}

## Section 5 (In-Process `native_tpu`): Portable Recovery Bundle & VM-Replacement Resume Simulation

Creates a fresh run, exports `bundle_pretraining_artifacts(..., include_resume_state=True)`, extracts `recovery/` into a **brand-new directory** (simulating a replacement VM), runs `restore_pretraining_report(..., restored_output_dir=...)`, and tests both exact resume and `extend_training_budget=True`.

In [ ]:
import zipfile
from trainlm import bundle_pretraining_artifacts, restore_pretraining_report


def _run_vm_recovery():
    base_dir = WORK_ROOT / "recovery-source"
    cfg = TrainLMPretrainingConfig(
        output_dir=base_dir,
        model_name_or_path="hf-internal-testing/tiny-random-gpt2",
        model_initialization="random",
        dataset_id="roneneldan/TinyStories",
        dataset_revision="main",
        train_split="train[:64]",
        validation_split="validation[:16]",
        max_train_examples=32,
        max_eval_examples=8,
        max_tokens=155,
        sequence_length=32,
        per_device_train_batch_size=1,
        logging_steps=2,
        accelerator="native_tpu",
        bf16=(PRECISION_MODE != "fp32"),
    )
    fresh = run_pretraining(cfg, show_progress=True, source_revision=SOURCE_REVISION)
    zip_path = bundle_pretraining_artifacts(fresh.report_path, include_resume_state=True)

    restored_dir = WORK_ROOT / "restored-vm-run"
    restored_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            if info.filename.startswith("recovery/") and not info.is_dir():
                rel = Path(info.filename).relative_to("recovery")
                target = restored_dir / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(zf.read(info.filename))

    restored = restore_pretraining_report(restored_dir / "report.json", restored_output_dir=restored_dir)
    exact_res = run_pretraining(
        restored.config,
        resume_from_checkpoint=restored.checkpoint_path,
        extend_training_budget=False,
        source_revision=SOURCE_REVISION,
        report_path=restored_dir / "exact-resume-report.json",
        export_path=restored_dir / "exact-resume-export",
    )

    from dataclasses import replace
    extended_cfg = replace(restored.config, max_tokens=restored.config.max_tokens + 120)
    ext_res = run_pretraining(
        extended_cfg,
        resume_from_checkpoint=restored.checkpoint_path,
        extend_training_budget=True,
        show_progress=True,
        source_revision=SOURCE_REVISION,
        report_path=restored_dir / "extended-report.json",
        export_path=restored_dir / "extended-export",
    )
    gen_text = verify_cpu_export_generation(ext_res.export_path)
    return {
        "fresh_steps": fresh.training.get("step"),
        "exact_resume_steps": exact_res.training.get("step"),
        "extended_steps": ext_res.training.get("step"),
        "extended_tokens": ext_res.training.get("supervised_tokens"),
        "generation_preview": gen_text[:120],
    }


record_check("vm_replacement_recovery_and_extend", TEST_VM_RECOVERY, _run_vm_recovery)

## Section 6 (In-Process `native_tpu`): Fine-Tuning Options Matrix (Currently Unvalidated on TPU)

Tests:
1. Finite Hub dataset (`dataset_revision="main"`, `bf16`, `gradient_checkpointing=True`, `report_to="jsonl"`)
2. Local JSONL (`train_file`/`validation_file`, `formatting_func`, `num_train_epochs=2`, `eval_strategy="epoch"`, `save_strategy="epoch"`, `load_best_model_at_end=True`, `early_stopping_patience=1`, `save_total_limit=2`)
3. Pretokenized HF `Dataset` (`data_format="tokens"` + `expected_tokenizer_vocab_sha256` + `loss_mask`)
4. Hub `streaming=True` + `restore_finetuning_config` exact cursor resume

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
from trainlm import (
    TrainLMFineTuningConfig,
    TrainLMTrainingArguments,
    restore_finetuning_config,
    run_finetuning,
    tokenizer_vocab_sha256,
)


def _run_ft_finite_hub():
    out = WORK_ROOT / "ft-hub"
    cfg = TrainLMFineTuningConfig(
        model_name_or_path="sshleifer/tiny-gpt2",
        dataset_id="roneneldan/TinyStories",
        dataset_revision="main",
        train_split="train[:32]",
        validation_split="validation[:16]",
        args=TrainLMTrainingArguments(
            output_dir=out,
            max_steps=3,
            sequence_length=64,
            per_device_train_batch_size=2,
            eval_strategy="no",
            save_strategy="no",
            logging_steps=1,
            report_to="jsonl",
            gradient_checkpointing=True,
            bf16=(PRECISION_MODE != "fp32"),
            accelerator="native_tpu",
        ),
    )
    res = run_finetuning(cfg)
    metrics_rows = (out / "metrics.jsonl").read_text(encoding="utf-8").splitlines()
    gen_text = verify_cpu_export_generation(res.model_path)
    return {"summary": res.summary(), "metrics_jsonl_lines": len(metrics_rows), "generation_preview": gen_text[:120]}


def _run_ft_local_epoch_best():
    d = WORK_ROOT / "ft-local-data"
    d.mkdir(parents=True, exist_ok=True)
    tf, vf = d / "train.jsonl", d / "val.jsonl"
    tf.write_text("\n".join(json.dumps({"q": f"Q{i}", "a": f"Answer {i} for fine-tuning epoch."}) for i in range(8)) + "\n", encoding="utf-8")
    vf.write_text("\n".join(json.dumps({"q": f"VQ{i}", "a": f"Validation {i} response."}) for i in range(4)) + "\n", encoding="utf-8")
    cfg = TrainLMFineTuningConfig(
        model_name_or_path="sshleifer/tiny-gpt2",
        train_file=tf,
        validation_file=vf,
        args=TrainLMTrainingArguments(
            output_dir=WORK_ROOT / "ft-local-epoch",
            max_steps=None,
            num_train_epochs=2,
            sequence_length=48,
            per_device_train_batch_size=2,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            early_stopping_patience=1,
            save_total_limit=2,
            accelerator="native_tpu",
            bf16=(PRECISION_MODE != "fp32"),
        ),
    )
    res = run_finetuning(cfg, formatting_func=lambda row: f"{row['q']} -> {row['a']}")
    return {"summary": res.summary(), "best_model_selection": res.best_model_selection}


def _run_ft_pretokenized():
    tok = AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
    rows = []
    for i in range(8):
        ids = tok(f"Tokenized fine-tuning sample number {i}.", add_special_tokens=False)["input_ids"]
        rows.append({"input_ids": ids, "labels": list(ids), "loss_mask": [1] * len(ids)})
    ds_train = Dataset.from_list(rows)
    ds_val = Dataset.from_list(rows[:2])
    cfg = TrainLMFineTuningConfig(
        model_name_or_path="sshleifer/tiny-gpt2",
        data_format="tokens",
        expected_tokenizer_vocab_sha256=tokenizer_vocab_sha256(tok),
        args=TrainLMTrainingArguments(
            output_dir=WORK_ROOT / "ft-tokens",
            max_steps=2,
            sequence_length=32,
            per_device_train_batch_size=2,
            eval_strategy="no",
            save_strategy="no",
            accelerator="native_tpu",
            bf16=(PRECISION_MODE != "fp32"),
        ),
    )
    res = run_finetuning(cfg, train_dataset=ds_train, eval_dataset=ds_val)
    return {"summary": res.summary()}


def _run_ft_streaming_resume():
    out = WORK_ROOT / "ft-streaming"
    cfg = TrainLMFineTuningConfig(
        model_name_or_path="sshleifer/tiny-gpt2",
        dataset_id="roneneldan/TinyStories",
        dataset_revision="main",
        streaming=True,
        args=TrainLMTrainingArguments(
            output_dir=out,
            max_steps=2,
            sequence_length=64,
            per_device_train_batch_size=2,
            eval_strategy="steps",
            eval_steps=2,
            save_strategy="steps",
            save_steps=2,
            max_eval_batches=2,
            dataloader_num_workers=0,
            accelerator="native_tpu",
            bf16=(PRECISION_MODE != "fp32"),
        ),
    )
    first = run_finetuning(cfg)
    restored = restore_finetuning_config(first.summary_path, max_steps=4)
    resumed = run_finetuning(restored, resume_from_checkpoint="latest")
    gen_text = verify_cpu_export_generation(resumed.model_path)
    return {"first_step": first.step, "resumed_step": resumed.step, "generation_preview": gen_text[:120]}


record_check("finetune_finite_hub_jsonl_ckpt", TEST_FINETUNING_OPTIONS, _run_ft_finite_hub)
record_check("finetune_local_epoch_best_early_stop", TEST_FINETUNING_OPTIONS, _run_ft_local_epoch_best)
record_check("finetune_pretokenized_vocab_sha", TEST_FINETUNING_OPTIONS, _run_ft_pretokenized)
record_check("finetune_streaming_cursor_resume", TEST_FINETUNING_STREAMING, _run_ft_streaming_resume)

## Section 7 (In-Process `native_tpu`): `PackedBinDataset` Epoch Schedules & `chunked_linear` Loss

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
from trainlm import PackedBinDataset, TrainLMTrainer, TrainLMTrainingArguments
from trainlm.data.writer import write_packed_binary_shard
from trainlm.optimization import inspect_dense_causal_lm
from trainlm.tasks import LinearCausalLMTrainingView, extract_hidden_state


def _run_epoch_and_chunked():
    root = WORK_ROOT / "trainer-advanced"
    model_dir = root / "tiny-gpt2"
    model_dir.mkdir(parents=True)
    GPT2LMHeadModel(GPT2Config(
        vocab_size=64, n_positions=16, n_ctx=16, n_embd=32, n_layer=1, n_head=2,
        bos_token_id=1, eos_token_id=2, pad_token_id=0, use_cache=False,
    )).save_pretrained(model_dir)
    docs = [[3 + ((r + o) % 60) for o in range(16)] for r in range(16)]
    train_dir, eval_dir = root / "train-bin", root / "eval-bin"
    write_packed_binary_shard(docs, train_dir, shard_id="train", vocab_size=64)
    write_packed_binary_shard(docs[:4], eval_dir, shard_id="eval", vocab_size=64)

    tr_ds = PackedBinDataset.from_directory(train_dir, sequence_length=16, split="train", seed=42)
    ev_ds = PackedBinDataset.from_directory(eval_dir, sequence_length=16, split="validation")
    try:
        trainer = TrainLMTrainer.from_pretrained(
            model_dir, train_dataset=tr_ds, eval_dataset=ev_ds,
            args=TrainLMTrainingArguments(
                output_dir=root / "epoch-out",
                max_steps=None, num_train_epochs=2, sequence_length=16,
                per_device_train_batch_size=2, gradient_accumulation_steps=3,
                eval_strategy="epoch", save_strategy="epoch", max_eval_batches=2,
                accelerator="native_tpu", bf16=(PRECISION_MODE != "fp32"),
            ),
        )
        epoch_state = trainer.train()
    finally:
        tr_ds.close()
        ev_ds.close()

    tr_res = PackedBinDataset.from_directory(train_dir, sequence_length=16, split="train", seed=42)
    ev_res = PackedBinDataset.from_directory(eval_dir, sequence_length=16, split="validation")
    try:
        resumed = TrainLMTrainer.from_pretrained(
            model_dir, train_dataset=tr_res, eval_dataset=ev_res,
            args=TrainLMTrainingArguments(
                output_dir=root / "epoch-resumed",
                max_steps=None, num_train_epochs=2, sequence_length=16,
                per_device_train_batch_size=2, gradient_accumulation_steps=3,
                eval_strategy="epoch", save_strategy="epoch", max_eval_batches=2,
                accelerator="native_tpu", bf16=(PRECISION_MODE != "fp32"),
            ),
        )
        resumed_state = resumed.train(resume_from_checkpoint=root / "epoch-out/checkpoint-3")
    finally:
        tr_res.close()
        ev_res.close()

    tr_chk = PackedBinDataset.from_directory(train_dir, sequence_length=16, split="train", seed=42)
    ev_chk = PackedBinDataset.from_directory(eval_dir, sequence_length=16, split="validation")
    try:
        chunked = TrainLMTrainer.from_pretrained(
            model_dir, train_dataset=tr_chk, eval_dataset=ev_chk,
            args=TrainLMTrainingArguments(
                output_dir=root / "chunked-out",
                max_steps=2, sequence_length=16, per_device_train_batch_size=2,
                eval_strategy="steps", eval_steps=2, max_eval_batches=2,
                loss_implementation="chunked_linear", logits_chunk_size=8,
                accelerator="native_tpu", bf16=(PRECISION_MODE != "fp32"),
            ),
            training_view_factory=lambda m: LinearCausalLMTrainingView(
                model=m,
                capabilities=inspect_dense_causal_lm(m),
                hidden_state_provider=lambda cur, batch: extract_hidden_state(
                    cur.transformer(input_ids=batch["input_ids"], attention_mask=batch.get("attention_mask"))
                ),
                provider_id="gpt2.transformer_hidden",
            ),
        )
        chunked_state = chunked.train()
    finally:
        tr_chk.close()
        ev_chk.close()

    return {
        "epoch_steps": epoch_state.step,
        "resumed_steps": resumed_state.step,
        "chunked_linear_steps": chunked_state.step,
        "chunked_linear_loss": chunked_state.loss,
    }


record_check("packed_epoch_and_chunked_linear", TEST_EPOCH_AND_CHUNKED_LOSS, _run_epoch_and_chunked)

## Section 8 (In-Process `native_tpu`): Real v5e-1 MFU Baseline & Bottleneck Profiler

Benchmarks three progressively larger models one by one on `TPU v5e-1` (16 GB HBM, 197 TFLOP/s BF16 peak) using tuned, OOM-safe configurations (`grad_accum=1`, larger microbatches, `chunked_linear` where needed, and `torch_compile=True` with eager comparison/fallback), separating **Window 1 (cold compile)** from **Windows 2–5 (steady-state MFU)**:
1. `HuggingFaceTB/SmolLM2-135M` (~2.2 GB static) — `seq_len=1024, batch_size=8` (8,192 tok/step), `grad_ckpt=False`, Eager vs `torch_compile=True` + CPU `torch.profiler` op breakdown
2. `HuggingFaceTB/SmolLM2-360M` (~5.8 GB static) — `seq_len=512, batch_size=8` (4,096 tok/step), `grad_ckpt=False`, `chunked_linear` (`chunk=1024`) to save logit HBM
3. `Qwen/Qwen2.5-0.5B` (494M, 151k vocab, ~7.9 GB static) — `seq_len=512, batch_size=4` (2,048 tok/step), `grad_ckpt=True`, `chunked_linear` (`chunk=1024`) to avoid 151k-vocab logit OOM

In [ ]:
import gc
from statistics import mean
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trainlm import TrainLMTrainer, TrainLMTrainingArguments
from trainlm.optimization import inspect_dense_causal_lm
from trainlm.tasks import LinearCausalLMTrainingView, extract_hidden_state


class _FixedShapeTokenDataset(TorchDataset):
    def __init__(self, *, vocab_size, seq_len, length=256):
        g = torch.Generator().manual_seed(123)
        self._rows = torch.randint(0, vocab_size, (length, seq_len), generator=g, dtype=torch.long)

    def __len__(self):
        return int(self._rows.shape[0])

    def __getitem__(self, idx):
        ids = self._rows[idx]
        return {"input_ids": ids, "labels": ids.clone()}


def _tpu_memory_gib():
    stats = {}
    tpu_mod = getattr(torch, "tpu", None)
    for key, attr in (("allocated_gib", "memory_allocated"), ("peak_gib", "max_memory_allocated")):
        fn = getattr(tpu_mod, attr, None)
        if callable(fn):
            try:
                stats[key] = round(float(fn()) / (1024 ** 3), 3)
            except Exception:
                pass
    return stats


def _cleanup_tpu():
    gc.collect()
    tpu_mod = getattr(torch, "tpu", None)
    for attr in ("empty_cache", "reset_peak_memory_stats"):
        fn = getattr(tpu_mod, attr, None)
        if callable(fn):
            try:
                fn()
            except Exception:
                pass


def _summarize_windows(perf):
    windows = perf.get("recent_windows") or []
    cold = windows[0] if windows else {}
    steady = windows[1:] if len(windows) > 1 else windows
    steady_tps = mean(w["train_tokens_per_second"] for w in steady) if steady else perf.get("tokens_per_second", 0.0)
    steady_mfu = mean(w.get("non_embedding_mfu", 0.0) for w in steady) if steady else perf.get("non_embedding_mfu", 0.0)
    steady_logits_mfu = mean(w.get("logits_inclusive_mfu", 0.0) for w in steady) if steady else perf.get("logits_inclusive_mfu", 0.0)
    steady_wait = mean(w.get("dataloader_wait_fraction", 0.0) for w in steady) if steady else perf.get("dataloader_wait_fraction", 0.0)
    return {
        "cold_window_1": {
            "tokens_per_second": round(float(cold.get("train_tokens_per_second", 0.0)), 2),
            "non_embedding_mfu_pct": round(float(cold.get("non_embedding_mfu", 0.0)) * 100.0, 2),
            "wall_seconds": round(float(cold.get("train_wall_seconds", 0.0)), 3),
        },
        "steady_state": {
            "windows_count": len(steady),
            "tokens_per_second": round(float(steady_tps), 2),
            "non_embedding_mfu_pct": round(float(steady_mfu) * 100.0, 2),
            "logits_inclusive_mfu_pct": round(float(steady_logits_mfu) * 100.0, 2),
            "dataloader_wait_fraction": round(float(steady_wait), 4),
        },
        "overall_tokens_per_second": round(float(perf.get("tokens_per_second", 0.0)), 2),
        "overall_mfu_pct": round(float(perf.get("non_embedding_mfu", 0.0) or 0.0) * 100.0, 2),
        "bottleneck_hint": perf.get("bottleneck_hint"),
    }


def _diagnose_bottleneck(summary, *, compile_enabled, grad_ckpt):
    steady = summary["steady_state"]
    mfu_pct = steady["non_embedding_mfu_pct"]
    wait = steady["dataloader_wait_fraction"]
    causes = []
    if wait >= 0.2:
        causes.append(f"high_dataloader_wait ({wait:.1%})")
    if not compile_enabled and mfu_pct < 20.0:
        causes.append("eager_python_dispatch_and_unfused_ops")
    if grad_ckpt:
        causes.append("activation_checkpoint_recompute_overhead")
    if mfu_pct < 15.0 and wait < 0.2:
        causes.append("small_batch_or_fp32_adamw_memory_bandwidth_bound")
    return causes or ["compute_or_kernel_bound"]


def _bench_model(label, model_id, *, seq_len, batch_size, grad_accum, max_steps, logging_steps, grad_ckpt, compile_enabled, use_chunked_loss, profile_ops=False):
    _cleanup_tpu()
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    vocab_size = int(getattr(model.config, "vocab_size", len(tok)))
    ds = _FixedShapeTokenDataset(vocab_size=vocab_size, seq_len=seq_len, length=max(128, batch_size * grad_accum * max_steps))
    out_dir = WORK_ROOT / f"mfu-{label}"

    view_factory = None
    if use_chunked_loss:
        def _causal_view(m):
            return LinearCausalLMTrainingView(
                model=m,
                capabilities=inspect_dense_causal_lm(m),
                hidden_state_provider=lambda cur, batch: extract_hidden_state(
                    cur.model(input_ids=batch["input_ids"], attention_mask=batch.get("attention_mask"))
                ),
                provider_id="hf.causal_model_hidden",
            )
        view_factory = _causal_view

    trainer = TrainLMTrainer(
        model=model,
        processing_class=tok,
        train_dataset=ds,
        args=TrainLMTrainingArguments(
            output_dir=out_dir,
            max_steps=max_steps,
            sequence_length=seq_len,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=grad_accum,
            learning_rate=1e-4,
            logging_steps=logging_steps,
            eval_strategy="no",
            save_strategy="no",
            bf16=True,
            accelerator="native_tpu",
            torch_compile=compile_enabled,
            gradient_checkpointing=grad_ckpt,
            loss_implementation="chunked_linear" if use_chunked_loss else "auto",
            logits_chunk_size=1024 if use_chunked_loss else None,
        ),
        training_view_factory=view_factory,
    )
    trainer.train()
    perf = trainer.performance_summary()
    mem = _tpu_memory_gib()
    summary = _summarize_windows(perf)
    summary["model_id"] = model_id
    summary["parameters"] = sum(p.numel() for p in model.parameters())
    summary["tokens_per_step"] = seq_len * batch_size * grad_accum
    summary["compile_enabled"] = compile_enabled
    summary["gradient_checkpointing"] = grad_ckpt
    summary["chunked_linear_loss"] = use_chunked_loss
    summary["hbm"] = mem
    summary["bottleneck_causes"] = _diagnose_bottleneck(summary, compile_enabled=compile_enabled, grad_ckpt=grad_ckpt)

    if profile_ops:
        try:
            with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU], record_shapes=False) as prof:
                batch = {k: v.to(trainer.runtime.device) for k, v in next(iter(trainer.engine.train_dataloader)).items()}
                with trainer.runtime.autocast():
                    out = trainer.model(**batch)
                    loss = out.loss
                loss.backward()
                trainer.optimizer.step()
                trainer.optimizer.zero_grad(set_to_none=True)
                trainer.runtime.synchronize()
            summary["top_cpu_ops"] = prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=10)
        except Exception as exc:
            summary["top_cpu_ops_error"] = f"{type(exc).__name__}: {exc}"

    del trainer, model
    _cleanup_tpu()
    return summary


def _bench_compile_with_fallback(label, model_id, **kwargs):
    if MFU_RUN_COMPILED:
        try:
            return _bench_model(f"{label}-compiled", model_id, compile_enabled=True, **kwargs)
        except Exception as exc:
            fallback = _bench_model(f"{label}-eager-fallback", model_id, compile_enabled=False, **kwargs)
            fallback["compile_error"] = f"{type(exc).__name__}: {exc}"
            return fallback
    return _bench_model(f"{label}-eager", model_id, compile_enabled=False, **kwargs)


def _run_mfu_baseline():
    report = {}
    report["smollm2_135m_eager"] = _bench_model(
        "smollm2-135m-eager", "HuggingFaceTB/SmolLM2-135M",
        seq_len=1024, batch_size=8, grad_accum=1, max_steps=25, logging_steps=5,
        grad_ckpt=False, compile_enabled=False, use_chunked_loss=False, profile_ops=True,
    )
    if MFU_RUN_COMPILED:
        try:
            report["smollm2_135m_compiled"] = _bench_model(
                "smollm2-135m-compiled", "HuggingFaceTB/SmolLM2-135M",
                seq_len=1024, batch_size=8, grad_accum=1, max_steps=25, logging_steps=5,
                grad_ckpt=False, compile_enabled=True, use_chunked_loss=False,
            )
        except Exception as exc:
            report["smollm2_135m_compiled"] = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}

    report["smollm2_360m"] = _bench_compile_with_fallback(
        "smollm2-360m", "HuggingFaceTB/SmolLM2-360M",
        seq_len=512, batch_size=8, grad_accum=1, max_steps=25, logging_steps=5,
        grad_ckpt=False, use_chunked_loss=True,
    )
    report["qwen2_5_0_5b"] = _bench_compile_with_fallback(
        "qwen25-05b", "Qwen/Qwen2.5-0.5B",
        seq_len=512, batch_size=4, grad_accum=1, max_steps=25, logging_steps=5,
        grad_ckpt=True, use_chunked_loss=True,
    )

    print("\n=== v5e-1 MFU Baseline Summary ===")
    for name, item in report.items():
        if "steady_state" not in item:
            print(f"  {name:24s} | FAILED: {item.get('error')}")
            continue
        s = item["steady_state"]
        c = item["cold_window_1"]
        print(
            f"  {name:24s} | compile={item['compile_enabled']} | tok/step={item['tokens_per_step']} "
            f"| cold={c['non_embedding_mfu_pct']:5.2f}% | steady={s['tokens_per_second']:8.1f} tok/s "
            f"| MFU={s['non_embedding_mfu_pct']:5.2f}% (logits={s['logits_inclusive_mfu_pct']:5.2f}%) "
            f"| hbm={item['hbm']} | causes={item['bottleneck_causes']}"
        )
    if "top_cpu_ops" in report["smollm2_135m_eager"]:
        print("\n--- Top CPU/Host Ops (SmolLM2-135M Eager) ---")
        print(report["smollm2_135m_eager"]["top_cpu_ops"])
    return report


record_check("mfu_baseline_and_bottlenecks", RUN_MFU_BASELINE, _run_mfu_baseline)

## Section 9 (Opt-In): Private Hugging Face Hub Publish & Live Reload

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


def _run_hub_publish():
    if LAST_EXPORT_RESULT is None:
        raise RuntimeError("No completed export result is available to publish.")
    published = LAST_EXPORT_RESULT.push_to_hub(HUB_REPO_ID, private=HUB_PRIVATE)
    tok = AutoTokenizer.from_pretrained(published.repo_id, revision=published.revision)
    mdl = AutoModelForCausalLM.from_pretrained(published.repo_id, revision=published.revision)
    inputs = tok("Once upon a time", return_tensors="pt")
    with torch.inference_mode():
        out = mdl.generate(**inputs, max_new_tokens=4, do_sample=False)
    return {
        "repo_id": published.repo_id,
        "revision": published.revision,
        "private": published.private,
        "reloaded_preview": tok.decode(out[0], skip_special_tokens=True)[:120],
    }


record_check("hub_publish_and_reload", bool(HUB_REPO_ID), _run_hub_publish)

## Section 10: Consolidated Evidence Report & Download

Run this cell even if earlier checks failed so you can review and share `trainlm_v5e1_full_validation_report.json`.

In [ ]:
FINAL_REPORT = {
    "schema_version": 1,
    "scope": "colab_v5e1_full_validation",
    "notebook_host": NOTEBOOK_HOST,
    "branch": BRANCH,
    "source_revision": SOURCE_REVISION,
    "installed_revision": INSTALLED_REVISION,
    "vm_machine_type_entered_by_owner": TPU_VM_MACHINE_TYPE,
    "packages": PACKAGES,
    "tpu_discovery": RESULTS.get("in_process_tpu_discovery"),
    "checks": {k: v for k, v in RESULTS.items() if k != "in_process_tpu_discovery"},
}

REPORT_PATH = Path.cwd() / "trainlm_v5e1_full_validation_report.json"
REPORT_PATH.write_text(json.dumps(FINAL_REPORT, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

print("=== Summary ===")
for name, item in FINAL_REPORT["checks"].items():
    print(f"  {name:38s} -> {item['status']}")
print(f"Report written to: {REPORT_PATH}")

try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(REPORT_PATH.name))
else:
    files.download(str(REPORT_PATH))